# R3 · 1 · Detector do tamanho da temporada

**Antes de rodar (uma vez só)**, em `MyDrive/dengue_V4/` (que já tem tudo das rodadas 1 e 2):
1. suba **`r3_lib.py`** (pasta `data_imdc_2026/RODADA3/`) e confira que o `r2_lib.py` mais recente está lá;
2. **crie à mão** as pastas `resultados_uf_r3`, `revisoes_r3`, `cache_r3` e `avaliacao_r3`.

**Complementar:** nada das rodadas anteriores muda. As previsões oficiais (corte de junho) vão para `resultados_uf_r3/uf/` (o CompUF4 e o Comp5 também as enxergam). As revisões de julho a outubro vão para `revisoes_r3/uf/`, fora da régua oficial. As tabelas vão para `avaliacao_r3/`, com data e hora no nome e uma cópia `_ultimo`.

**Retomada:** se a internet ou a luz cair, reconecte e **rode tudo de novo**. Cada etapa grava o seu arquivo (variáveis, um arquivo por método × corte do detector, parâmetros em json, um arquivo por modelo × temporada, um PNG por UF) e o que já existe é pulado.

**Dados externos (opcional):** para testar sinais que não estão no FTP (previsão de ENSO do IRI, sorotipo, vacinação, positividade de exames), ponha em `cache_r3/externos_r3.csv` colunas `uf_code`, `season` e, se quiser, `corte` (`jun`/`jul`/`ago`/`set`/`out`), mais uma coluna por sinal. Cada valor deve ser o que se sabia **naquele corte**. Apague `cache_r3/detector_*` e `cache_r3/sinais_detector.json` para refazer.

CPU comum, ~10–20 min.

**O que é:** uma linha por UF × temporada × corte. O alvo é o log do tamanho da temporada (SE41→SE40) **relativo ao normal da UF** (média das temporadas já terminadas). Os sinais são o que se sabe no corte: casos e espalhamento das últimas 4–13 semanas comparados com a mesma época dos anos anteriores, tendência, vizinhos e macrorregião, ENSO/IOD/PDO e a variação de 12 semanas, tamanho das últimas temporadas e anos desde a última grande.

**Sem vazamento:** os sinais de cada corte são escolhidos só com temporadas até 2021. Para prever a temporada *s*, cada método treina só com temporadas que já tinham terminado no corte. Métodos: regressão ridge (com faixa pelos resíduos), regressão quantílica linear, LightGBM quantílico pequeno e a média dos três. Por corte, fica o de menor erro (pinball) na afinação.

**Cortes:** jun = SE25 (regra oficial); jul = SE30; ago = SE34; set = SE38; out = SE40 (véspera do início da temporada).


In [ ]:
NOTEBOOK = 'r3_detector'

# @title 1 · Drive, arquivos e pacotes
import os, sys, shutil, time, glob, subprocess, json, traceback
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_DRIVE, PASTA_LOCAL, NO_COLAB = '/content/drive/MyDrive/dengue_V4', '/content/r3', True
except ImportError:
    PASTA_DRIVE = PASTA_LOCAL = os.path.abspath('.'); NO_COLAB = False
if NO_COLAB:
    os.makedirs(PASTA_LOCAL, exist_ok=True)
    for item in ('v4_lib.py', 'comp_lib.py', 'comp_modelos.py', 'uf_lib.py', 'semanal_lib.py', 'r2_lib.py', 'r3_lib.py',
                 'dados', 'dados_uf', 'referencias_imdc3'):
        o, d = os.path.join(PASTA_DRIVE, item), os.path.join(PASTA_LOCAL, item)
        assert os.path.exists(o), f'Faltou subir {item} para {PASTA_DRIVE}'
        shutil.copytree(o, d, dirs_exist_ok=True) if os.path.isdir(o) else shutil.copy(o, d)
sys.path.insert(0, PASTA_LOCAL)
PASTA_R3 = os.path.join(PASTA_DRIVE, 'resultados_uf_r3')
PASTA_REV = os.path.join(PASTA_DRIVE, 'revisoes_r3')
PASTA_C3 = os.path.join(PASTA_DRIVE, 'cache_r3')
PASTA_AV3 = os.path.join(PASTA_DRIVE, 'avaliacao_r3')
PASTA_AV2 = os.path.join(PASTA_DRIVE, 'avaliacao_r2')
PASTA_COMP = os.path.join(PASTA_DRIVE, 'resultados_competicao')
PASTA_EXT = os.path.join(PASTA_DRIVE, 'resultados_externos')
for p in (os.path.join(PASTA_R3, 'uf'), os.path.join(PASTA_REV, 'uf'), PASTA_C3, PASTA_AV3):
    os.makedirs(p, exist_ok=True)
import importlib, v4_lib as V, comp_lib as L, comp_modelos as CM, uf_lib as U, semanal_lib as S, r2_lib as R2, r3_lib as R3
for m in (V, L, CM, U, S, R2, R3):
    importlib.reload(m)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 200)
ctx = U.ContextoUF(os.path.join(PASTA_LOCAL, 'dados'), os.path.join(PASTA_LOCAL, 'referencias_imdc3'),
                   os.path.join(PASTA_LOCAL, 'dados_uf'))
D = ctx.Du
TUNING, TESTE = U.TUNING, U.TESTE
REFAZER = False
def detector_escolhido():
    esc = R3.json_le(os.path.join(PASTA_C3, 'detector_escolhido.json'))
    assert esc, 'Rode antes o R3_1_Detector.'
    P = pd.concat([pd.read_parquet(os.path.join(PASTA_C3, f'detector_{m}_{k}.parquet')) for k, m in esc.items()], ignore_index=True)
    return esc, P


In [ ]:
# @title 2 · Variáveis e alvo (cache; retoma)
arq_ext = os.path.join(PASTA_C3, 'externos_r3.csv')
EXT = pd.read_csv(arq_ext) if os.path.exists(arq_ext) else None
print('externos:', None if EXT is None else list(EXT.columns))
X = R3.tabela_detector(ctx, list(range(2011, 2026)), PASTA_C3, externos=EXT)
print(X.shape)
X.groupby('corte').alvo.describe().round(2)


In [ ]:
# @title 3 · Sinais escolhidos só com temporadas ≤ 2021 (correlação de Spearman com o alvo)
ARQ_SIN = os.path.join(PASTA_C3, 'sinais_detector.json')
SINAIS = {k: R3.selecionar_sinais(X, k, ate_temporada=max(TUNING), arq=ARQ_SIN) for k in R3.CORTES}
pd.DataFrame({k: pd.Series(v['rho']) for k, v in R3.json_le(ARQ_SIN).items()}).round(2)


In [ ]:
# @title 4 · Previsões do detector (janela expansiva; um arquivo por método × corte) e avaliação
P = R3.prever_detector(ctx, X, SINAIS, list(range(2014, 2026)), PASTA_C3)
TD = R3.avaliar_detector(P)
R3.salvar_tabela(TD, PASTA_AV3, 'detector_avaliacao')
ESC = R3.escolher_detector(TD)
R3.json_grava(ESC, os.path.join(PASTA_C3, 'detector_escolhido.json'))
print('escolhido por corte (menor pinball na afinação):', ESC)
TD[TD.grupo != 'anterior'].pivot_table(index=['corte', 'metodo'], columns='grupo',
                                       values=['pinball', 'corr_mediana', 'IC90', 'sensibilidade', 'precisao']).round(2)


In [ ]:
# @title 5 · O que o detector disse em junho de cada temporada de teste (tamanho × o normal da UF)
DJ = pd.concat([P[(P.metodo == m) & (P.corte == k)] for k, m in ESC.items()])
DJ = DJ[(DJ.corte == 'jun') & DJ.season.isin(TESTE)].copy()
DJ['uf'] = DJ.uf_code.map(R2.SIGLAS)
DJ['real_x'], DJ['previsto_x'] = np.exp(DJ.alvo), np.exp(DJ.pred)
DJ['faixa90_x'] = [f'{np.exp(a):.1f}–{np.exp(b):.1f}' for a, b in zip(DJ.lower_90, DJ.upper_90)]
TAB = DJ.pivot_table(index='uf', columns='season', values=['real_x', 'previsto_x', 'p_grande']).round(2)
R3.salvar_tabela(TAB.reset_index(), PASTA_AV3, 'detector_junho_por_uf')
TAB
